# Pandas Part 2: Cleaning and Transforming Data in Pandas

In real analytics and data science work, the hardest part is often **not modeling**. It is cleaning the data well enough that your analysis can be trusted.

This session is built around a realistic truth:

> Raw data is rarely analysis-ready.

Analysts routinely face:
- missing values
- duplicates
- inconsistent text labels
- messy dates
- wrong data types
- invalid numbers
- placeholder values like `"N/A"` or `"unknown"`
- columns that look numeric but are stored as strings

This notebook is designed to help students build a real workflow for transforming messy business data into a usable analytical dataset.

---

## Business scenario

You are on the analytics team for a specialty beverage retailer.  
Management wants a trustworthy monthly performance dashboard and a cleaned dataset that can be used later for:
- visualization
- customer analysis
- machine learning
- time series work

However, the raw extract has many quality problems.

Your task is to:
1. audit the data
2. identify problems
3. clean the data
4. create new useful variables
5. produce a cleaned analytical table

---

## Learning objectives

By the end of this session, students should be able to:

- Identify common data quality issues in tabular data
- Diagnose missing, duplicated, inconsistent, and invalid values
- Clean string columns using Pandas string methods
- Convert columns to correct numeric and datetime types
- Handle missing values in practical ways
- Remove or flag duplicate records
- Create transformed columns for analysis
- Build a repeatable data-cleaning workflow
- Explain and justify cleaning decisions in business terms




## Imports and display settings

This session keeps the library stack simple:
- **pandas** for data cleaning and transformation
- **numpy** for missing values and numerical helpers

A key lesson: strong data cleaning usually comes from a **small set of tools used thoughtfully**, not from memorizing dozens of exotic functions.


In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Pandas version:", pd.__version__)

Pandas version: 3.0.0


## Load the messy dataset

This session uses a deliberately messy CSV file:

`retail_data_messy.csv`

It includes realistic issues such as:
- missing values
- blank strings and placeholder labels
- duplicates
- inconsistent capitalization
- extra whitespace
- numeric columns stored as strings
- mixed date formats
- suspicious negative shipping costs
- zero quantities
- missing revenue values

The goal is not just to "fix everything."  
The real goal is to learn how to make **good analytical decisions**.


In [2]:
raw_df = pd.read_csv("retail_data_messy.csv")
raw_df.head(10)

,order_id,order_date,customer_id,segment,channel,country,city,region,category,product_name,quantity,unit_price,discount,shipping_cost,revenue
0,O100002,2024-04-01,C10061,Consumer,Online,Germany,Munich,Europe,Merch,Ceramic Mug,2,14.0,0.0,10.36,28.00
1,O100004,2024-05-07,C10147,Small Business,Marketplace,Australia,Brisbane,APAC,Merch,Tote Bag,3,18.0,0.0,10.78,54.00
2,O100005,2024-03-15,C10303,Corporate,Marketplace,Australia,Melbourne,APAC,Bakery,Biscotti Jar,2,11.5,0.0,-9.33,23.00
3,O100006,2024-01-18,C10309,Small Business,Store Pickup,United States,San Jose,North America,Coffee,Colombian Beans,4,16.5,0.0,6.87,66.00
4,O100009,2024-03-26,C10443,Small Business,Marketplace,France,Lyon,Europe,Tea,Matcha Tin,2,19.0,0.0,2.57,38.00
5,O100011,2024-04-12,C10074,Consumer,Marketplace,Germany,Munich,Europe,Equipment,Pour Over Kit,2,34.0,0.0,8.15,68.00
6,O100012,2024-06-16,C10139,Consumer,Online,United Kingdom,Birmingham,Europe,Coffee,Colombian Beans,5,16.5,0.05,4.84,78.38
7,O100014,2024-05-06,C10264,Small Business,Online,United Kingdom,London,Europe,Bakery,Muffin Box,3,13.0,0.0,2.73,39.00
8,O100015,2024-03-03,C10290,Consumer,Online,United States,Chicago,North America,Coffee,Espresso Blend,3,17.5,0.0,8.80,52.50
9,O100018,2024-03-22,C10629,Consumer,Online,Australia,Melbourne,APAC,Bakery,Biscotti Jar,4,11.5,0.0,6.99,46.00


## Start with a data quality audit

Before cleaning anything, inspect the raw data carefully.

A disciplined workflow begins with questions like:
- What does one row represent?
- Which columns should be numeric?
- Which columns should be dates?
- Which columns are categorical?
- What values already look suspicious?
- What kinds of issues are likely to break analysis later?


In [3]:
raw_df.shape

(4532, 15)

In [4]:
raw_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4532 entries, 0 to 4531
Data columns (total 15 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       4532 non-null   str    
 1   order_date     4532 non-null   str    
 2   customer_id    4532 non-null   str    
 3   segment        4321 non-null   str    
 4   channel        4453 non-null   str    
 5   country        4532 non-null   str    
 6   city           4429 non-null   str    
 7   region         4532 non-null   str    
 8   category       4532 non-null   str    
 9   product_name   4496 non-null   str    
 10  quantity       4532 non-null   int64  
 11  unit_price     4532 non-null   str    
 12  discount       4532 non-null   str    
 13  shipping_cost  4353 non-null   float64
 14  revenue        4520 non-null   float64
dtypes: float64(2), int64(1), str(12)
memory usage: 531.2 KB


In [5]:
raw_df.sample(10, random_state=42)

,order_id,order_date,customer_id,segment,channel,country,city,region,category,product_name,quantity,unit_price,discount,shipping_cost,revenue
3576,O107108,2024-03-12,C10600,Consumer,Online,Canada,Toronto,North America,Coffee,Cold Brew Pack,5,14.0,0.0,7.43,70.00
4509,O107191,2024-04-19,C10019,Consumer,Store Pickup,Canada,Vancouver,North America,Merch,T-Shirt,2,26.0,0.0,6.88,52.00
151,O100280,2024-04-07,C10551,NaN,NaN,Germany,Berlin,Europe,Merch,Travel Mug,2,22.0,0.0,NaN,44.00
2099,O104183,2024-01-09,C10091,Consumer,Online,United States,San Jose,North America,Bakery,Cookie Pack,3,9.0,0.05,3.74,25.65
3015,O105986,2024-01-20,C10624,Small Business,Online,United Kingdom,Birmingham,Europe,Equipment,Burr Grinder,5,79.0,0.0,7.28,395.00
465,O100905,2024-02-24,C10117,Corporate,Store Pickup,United States,New York,North America,Tea,Earl Grey,3,11.0,0.0,4.64,33.00
668,O101320,2024-03-17,C10179,Small Business,Store Pickup,Canada,Montreal,North America,Bakery,Cookie Pack,1,9.0,0.0,10.20,9.00
4156,O108298,2024-01-10,C10042,Consumer,Marketplace,Canada,Toronto,North America,Coffee,Colombian Beans,1,16.5,0.0,NaN,16.50
721,O101426,2024-01-03,C10235,Corporate,Online,United States,San Jose,North America,Bakery,Croissant Box,4,15.0,0.0,8.47,60.00
1554,O103091,2024-01-08,C10119,Consumer,Store Pickup,Canada,Toronto,North America,Equipment,Milk Frother,3,$24.0,0.0,5.33,72.00


### Questions
- Which columns appear messy immediately?
- Which columns should probably change type?
- Which columns may contain inconsistent labels?
- Which values would make dashboards inaccurate if left uncleaned?


## Create a working copy

Always avoid cleaning directly on the original raw DataFrame.

A good habit is:

```python
df = raw_df.copy()
```

This preserves the raw extract for auditing and reproducibility.


In [6]:
df = raw_df.copy()
df.head()

,order_id,order_date,customer_id,segment,channel,country,city,region,category,product_name,quantity,unit_price,discount,shipping_cost,revenue
0,O100002,2024-04-01,C10061,Consumer,Online,Germany,Munich,Europe,Merch,Ceramic Mug,2,14.0,0.0,10.36,28.00
1,O100004,2024-05-07,C10147,Small Business,Marketplace,Australia,Brisbane,APAC,Merch,Tote Bag,3,18.0,0.0,10.78,54.00
2,O100005,2024-03-15,C10303,Corporate,Marketplace,Australia,Melbourne,APAC,Bakery,Biscotti Jar,2,11.5,0.0,-9.33,23.00
3,O100006,2024-01-18,C10309,Small Business,Store Pickup,United States,San Jose,North America,Coffee,Colombian Beans,4,16.5,0.0,6.87,66.00
4,O100009,2024-03-26,C10443,Small Business,Marketplace,France,Lyon,Europe,Tea,Matcha Tin,2,19.0,0.0,2.57,38.00


## Audit missing values

Missing data is often the first thing analysts check.

But missingness is not just a technical issue. It can mean:
- data was not collected
- data was unavailable
- data was not applicable
- data was entered inconsistently
- a system integration failed

So the question is not only **"where are the missing values?"** but also **"what do they mean?"**


In [7]:
df.isna().sum().sort_values(ascending=False)

segment          211
shipping_cost    179
city             103
channel           79
product_name      36
revenue           12
customer_id        0
country            0
order_id           0
order_date         0
region             0
quantity           0
category           0
discount           0
unit_price         0
dtype: int64

In [8]:
(df.isna().mean() * 100).sort_values(ascending=False).round(2)

segment         4.66
shipping_cost   3.95
city            2.27
channel         1.74
product_name    0.79
revenue         0.26
customer_id     0.00
country         0.00
order_id        0.00
order_date      0.00
region          0.00
quantity        0.00
category        0.00
discount        0.00
unit_price      0.00
dtype: float64

### Mini exercise

Inspect missing values and answer:
1. Which columns have the most missing values?
2. Which missing values might be tolerable?
3. Which missing values would prevent downstream analysis?
4. Which columns might need a missing-value flag instead of direct deletion?


##  Detect hidden missing values

A major real-world issue is that missing values often do **not** appear as `NaN`.

Instead, they may show up as:
- empty string `""`
- single blank `" "`
- `"unknown"`
- `"N/A"`
- `"null"`

These need to be treated explicitly.


In [41]:
object_cols = df.select_dtypes(include="object").columns.tolist() # selecting only object columns
object_cols

C:\Users\hlu\AppData\Local\Temp\ipykernel_5056\4223095510.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = df.select_dtypes(include="object").columns.tolist() # selecting only object columns


['order_id', 'customer_id', 'order_month', 'revenue_band']

In [10]:
for col in ["city", "segment", "channel"]:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).head(15))


--- city ---
city
Chicago       352
San Jose      350
Seattle       338
New York      326
Austin        305
Manchester    286
London        282
Birmingham    262
Vancouver     185
Montreal      184
Berlin        184
Hamburg       180
Munich        179
Toronto       170
Paris         152
Name: count, dtype: int64

--- segment ---
segment
Consumer          2730
Small Business    1052
Corporate          524
NaN                211
                     8
unknown              7
Name: count, dtype: int64

--- channel ---
channel
Online          2705
Marketplace      901
Store Pickup     804
NaN               79
online            21
marketplace        7
                   7
unknown            6
store pickup       2
Name: count, dtype: int64


### Challenge
We often think `isna()` is enough. It is not.

A string such as `"unknown"` is not missing to Pandas by default, but it may still be analytically useless.


In [11]:
placeholders = ["", " ", "unknown", "N/A", "n/a", "Unknown", "NULL", "null"]

for col in ["city", "segment", "channel"]:
    count_placeholder = df[col].astype(str).isin(placeholders).sum()
    print(col, "placeholder-like values:", count_placeholder)

city placeholder-like values: 20
segment placeholder-like values: 15
channel placeholder-like values: 13


## Standardize placeholder values to missing

A clean strategy is to convert placeholder values into proper missing values first.


In [43]:
for col in ["city", "segment", "channel"]:
    df[col] = df[col].replace(["", " ", "unknown", "N/A", "n/a", "Unknown", "NULL", "null"], np.nan)

df[["city", "segment", "channel"]].sample(10, random_state=1)

,city,segment,channel
44,Toronto,Consumer,Marketplace
840,Montreal,Consumer,Store Pickup
2027,Hamburg,Small Business,Store Pickup
3517,Birmingham,Small Business,Online
1185,Brisbane,Consumer,Store Pickup
4117,London,Consumer,Online
90,Munich,Consumer,Store Pickup
2011,Lyon,Small Business,Online
4376,Seattle,Consumer,Online
1080,Berlin,Consumer,Online


In [44]:
df.isna().sum().sort_values(ascending=False)

segment             225
shipping_cost       197
city                123
channel              92
order_date           55
order_month          55
order_year           55
product_name         36
revenue              12
order_id              0
country               0
customer_id           0
category              0
region                0
discount              0
unit_price            0
quantity              0
is_zero_quantity      0
is_return             0
net_unit_price        0
abs_quantity          0
revenue_band          0
dtype: int64

## Cleaning text: strip whitespace

Whitespace issues are subtle and common.

Examples:
- `"United States"`
- `" UNITED STATES "`
- `"United States "`

These will behave like different values unless standardized.


In [45]:
for col in ["country", "city", "segment", "channel", "region", "category", "product_name"]:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip()

df[["country", "category", "channel"]].head(10)

,country,category,channel
0,Germany,Merch,Online
1,Australia,Merch,Marketplace
2,Australia,Bakery,Marketplace
3,United States,Coffee,Store Pickup
4,France,Tea,Marketplace
5,Germany,Equipment,Marketplace
6,United Kingdom,Coffee,Online
7,United Kingdom,Bakery,Online
8,United States,Coffee,Online
9,Australia,Bakery,Online


## Standardize capitalization and category labels

After whitespace cleanup, category values may still be inconsistent due to casing differences.

Examples:
- `Coffee`
- `cOFFEE`
- `COFFEE`

We need to harmonize them.


In [15]:
print("Before standardization:")
print(df["category"].value_counts(dropna=False).head(20))

Before standardization:
category
Coffee       1255
Bakery        926
Equipment     822
Merch         783
Tea           701
cOFFEE         12
tEA             8
bAKERY          7
eQUIPMENT       5
merch           4
coffee          3
mERCH           3
equipment       2
tea             1
Name: count, dtype: Int64


In [48]:
df["country"] = df["country"].str.title()
df["city"] = df["city"].str.title()
df["segment"] = df["segment"].str.title()
df["region"] = df["region"].str.upper()
df["channel"] = df["channel"].str.title()
df["category"] = df["category"].str.title()

print("After standardization:")
print(df["category"].value_counts(dropna=False).head(20))

After standardization:
category
Coffee       1261
Bakery        924
Equipment     826
Merch         788
Tea           705
Name: count, dtype: Int64


### Question

Should every text field be title-cased?

Not always. For example:
- acronyms may need all-caps
- product names may have preferred branding
- IDs should not be altered casually

Cleaning should be **intentional**, not mechanical.


## Cleaning mixed-format numeric columns

Real raw files often contain:
- currency symbols like `$18.50`
- percentages like `10%`
- commas like `1,250`
- blank strings
- inconsistent decimal formatting

Let's inspect the suspect columns.


In [17]:
df[["unit_price", "discount", "shipping_cost"]].sample(12, random_state=5)

,unit_price,discount,shipping_cost
2589,17.5,0.0,4.15
3727,14.0,0.0,3.57
2751,12.0,15%,4.49
1328,28.0,0.15,4.36
621,26.0,0.0,5.89
390,28.0,0.0,9.64
3319,11.0,0%,8.57
3581,14.0,0.0,3.61
1023,14.0,0%,8.11
668,9.0,0.0,10.20


### Convert `unit_price` to numeric

Some values contain a dollar sign.  
We remove the symbol and then convert to numeric.


In [49]:
df["unit_price"] = (
    df["unit_price"]
      .astype("string")
      .str.replace("$", "", regex=False)
) # Remove $ sign from unit_price column

df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce") # Convert unit_price to numeric, coercing errors to NaN
df["unit_price"].head()

0   14.00
1   18.00
2   11.50
3   16.50
4   19.00
Name: unit_price, dtype: Float64

### Convert `discount` to numeric fraction

Some values look like:
- `0.05`
- `10%`

We need a consistent representation.  
For this course, we will store discount as a **fraction**:
- 5% becomes `0.05`
- 10% becomes `0.10`


In [53]:
discount_str = df["discount"].astype("string").str.strip() # Convert discount to string and strip whitespace

is_percent = discount_str.str.contains("%", na=False) # Boolean mask for percentage values

discount_clean = discount_str.str.replace("%", "", regex=False) # Remove % sign from discount column

discount_num = pd.to_numeric(discount_clean, errors="coerce") # Convert to numeric, coercing errors to NaN

discount_num = np.where(is_percent, discount_num / 100, discount_num) # Convert percentage values to decimal

df["discount"] = pd.to_numeric(discount_num, errors="coerce") # Assign cleaned discount values back to the DataFrame, ensuring numeric type

df["discount"].head(12)

0    0.00
1    0.00
2    0.00
3    0.00
4    0.00
5    0.00
6    0.05
7    0.00
8    0.00
9    0.00
10   0.00
11   0.00
Name: discount, dtype: float64

### Check numeric conversion results


In [54]:
df[["unit_price", "discount", "shipping_cost", "revenue"]].info()

<class 'pandas.DataFrame'>
Index: 4504 entries, 0 to 4509
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   unit_price     4504 non-null   Float64
 1   discount       4504 non-null   float64
 2   shipping_cost  4307 non-null   float64
 3   revenue        4492 non-null   float64
dtypes: Float64(1), float64(3)
memory usage: 180.3 KB


## Convert dates safely

Date columns often arrive in mixed formats:
- `2024-01-05`
- `01/05/2024`
- `2024/01/05`

A strong analyst converts dates carefully and checks for failures.


In [21]:
df["order_date"].head(12)

0     2024-04-01
1     2024-05-07
2     2024-03-15
3     2024-01-18
4     2024-03-26
5     2024-04-12
6     2024-06-16
7     2024-05-06
8     2024-03-03
9     2024-03-22
10    2024-01-26
11    2024-02-24
Name: order_date, dtype: str

In [22]:
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
df["order_date"].head(12)

0    2024-04-01
1    2024-05-07
2    2024-03-15
3    2024-01-18
4    2024-03-26
5    2024-04-12
6    2024-06-16
7    2024-05-06
8    2024-03-03
9    2024-03-22
10   2024-01-26
11   2024-02-24
Name: order_date, dtype: datetime64[us]

In [23]:
df["order_date"].isna().sum()

np.int64(56)

### Question

If some dates fail conversion and become missing, what should you do?

Possible actions:
- inspect the failed records
- fix patterns manually
- drop if very rare and not important
- ask the data owner if source formatting changed


## Detect duplicates

Duplicates can distort counts, sums, and averages.

But duplicates are tricky:
- some are accidental
- some are legitimate repeated transactions
- some are duplicates only across certain columns

Start with exact duplicates, then think more carefully.


In [55]:
df.duplicated().sum()

np.int64(0)

In [56]:
df[df.duplicated()].head(10)

,order_id,order_date,customer_id,segment,channel,country,city,region,category,product_name,quantity,unit_price,discount,shipping_cost,revenue,is_zero_quantity,is_return,order_year,order_month,net_unit_price,abs_quantity,revenue_band


### Remove exact duplicates

For this teaching dataset, we will remove exact duplicates.
In real projects, always document and justify this decision.


In [26]:
before_rows = len(df)
df = df.drop_duplicates()
after_rows = len(df)

print("Rows before:", before_rows)
print("Rows after :", after_rows)
print("Removed    :", before_rows - after_rows)

Rows before: 4532
Rows after : 4504
Removed    : 28


## Validate business-rule problems

Not all dirty data is missing or duplicated.

Some values may be invalid according to business logic.

Examples:
- negative shipping cost
- zero quantity
- negative quantity that may reflect returns
- missing revenue
- revenue inconsistent with price × quantity × discount

A good cleaning session includes **business validation**, not just formatting fixes.


In [57]:
df[["quantity", "unit_price", "discount", "shipping_cost", "revenue"]].describe().T

,count,mean,std,min,25%,50%,75%,max
quantity,"4,504.00",2.91,1.54,-2.00,2.00,3.00,4.00,5.00
unit_price,"4,504.00",20.21,14.27,8.55,13.00,16.62,22.00,82.95
discount,"4,504.00",0.02,0.04,0.00,0.00,0.00,0.00,0.15
shipping_cost,"4,307.00",7.48,2.89,2.50,5.01,7.43,9.98,12.49
revenue,"4,492.00",58.29,55.89,-165.90,27.00,47.02,72.00,414.75


In [58]:
print("Negative shipping cost:", (df["shipping_cost"] < 0).sum())
print("Zero quantity:", (df["quantity"] == 0).sum())
print("Negative quantity:", (df["quantity"] < 0).sum())
print("Missing revenue:", df["revenue"].isna().sum())

Negative shipping cost: 0
Zero quantity: 15
Negative quantity: 93
Missing revenue: 12


### Question

Which of the following are automatically wrong, and which may be legitimate?
- negative shipping cost
- zero quantity
- negative quantity
- missing revenue

This is the difference between **data cleaning** and **blind rule enforcement**.


## Handling suspicious values

For this session, we will use a practical strategy:
- negative shipping cost → set to missing, then impute
- zero quantity → flag as suspicious
- negative quantity → keep, because it may indicate returns
- missing revenue → recompute when possible


In [59]:
df.loc[df["shipping_cost"] < 0, "shipping_cost"] = np.nan # Set negative shipping_cost values to NaN
df["is_zero_quantity"] = df["quantity"].eq(0) # Create is_zero_quantity column that is True when quantity is 0, False otherwise
df["is_return"] = df["quantity"].lt(0) # Create is_return column that is True when quantity is less than 0, False otherwise

df[["quantity", "shipping_cost", "is_zero_quantity", "is_return"]].head(10)

,quantity,shipping_cost,is_zero_quantity,is_return
0,2,10.36,False,False
1,3,10.78,False,False
2,2,NaN,False,False
3,4,6.87,False,False
4,2,2.57,False,False
5,2,8.15,False,False
6,5,4.84,False,False
7,3,2.73,False,False
8,3,8.80,False,False
9,4,6.99,False,False


## Recompute revenue from business logic

If quantity, unit price, and discount are available, revenue can be recomputed as:

```python
quantity * unit_price * (1 - discount)
```

This is a valuable example of using domain logic to repair missing data.


In [ ]:
revenue_calc = df["quantity"] * df["unit_price"] * (1 - df["discount"])
revenue_calc = revenue_calc.round(2)

df["revenue_cleaned"] = df["revenue"].fillna(revenue_calc) # Fill missing revenue values with calculated revenue
df[["quantity", "unit_price", "discount", "revenue", "revenue_cleaned"]].head(12)

In [ ]:
df["revenue"] = df["revenue_cleaned"]
df = df.drop(columns=["revenue_cleaned"])

df["revenue"].isna().sum()

## Handle missing values strategically

There is no single correct rule for missing values.

Common choices:
- drop rows
- fill with mean/median/mode
- fill with business default
- use `"Unknown"`
- create a missing flag
- leave missing if the downstream method can handle it

Here we will use practical decisions:
- `segment`, `channel`, `city` → fill with `"Unknown"`
- `shipping_cost` → fill with median
- `product_name` → drop rows if missing because product detail matters


In [ ]:
df["segment"] = df["segment"].fillna("Unknown")
df["channel"] = df["channel"].fillna("Unknown")
df["city"] = df["city"].fillna("Unknown")

shipping_median = df["shipping_cost"].median()
df["shipping_cost"] = df["shipping_cost"].fillna(shipping_median)

before = len(df)
df = df.dropna(subset=["product_name"])
after = len(df)

print("Dropped rows with missing product_name:", before - after)

In [ ]:
df.isna().sum().sort_values(ascending=False)

### Mini exercise

Explain why these decisions are reasonable or not:
1. Fill missing `segment` with `"Unknown"`
2. Fill missing `shipping_cost` with the median
3. Drop rows with missing `product_name`

Could there be alternative choices?


## Create transformed features

Cleaning is only half the story.  
Transformation creates variables that are more useful for analysis.

Examples:
- month and year fields from dates
- net price after discount
- absolute quantity
- order value buckets
- flags for returns and missingness


In [ ]:
df["order_year"] = df["order_date"].dt.year # Extract year from order_date and create order_year column
df["order_month"] = df["order_date"].dt.to_period("M").astype(str) # Extract month from order_date, convert to period (YYYY-MM) and then to string
df["net_unit_price"] = (df["unit_price"] * (1 - df["discount"])).round(2) # Calculate net unit price
df["abs_quantity"] = df["quantity"].abs() # Calculate absolute quantity

df[["order_date", "order_year", "order_month", "unit_price", "discount", "net_unit_price", "quantity", "abs_quantity"]].head(10)

,order_date,order_year,order_month,unit_price,discount,net_unit_price,quantity,abs_quantity
0,2024-04-01,"2,024.00",2024-04,14.00,0.00,14.00,2,2
1,2024-05-07,"2,024.00",2024-05,18.00,0.00,18.00,3,3
2,2024-03-15,"2,024.00",2024-03,11.50,0.00,11.50,2,2
3,2024-01-18,"2,024.00",2024-01,16.50,0.00,16.50,4,4
4,2024-03-26,"2,024.00",2024-03,19.00,0.00,19.00,2,2
5,2024-04-12,"2,024.00",2024-04,34.00,0.00,34.00,2,2
6,2024-06-16,"2,024.00",2024-06,16.50,0.05,15.68,5,5
7,2024-05-06,"2,024.00",2024-05,13.00,0.00,13.00,3,3
8,2024-03-03,"2,024.00",2024-03,17.50,0.00,17.50,3,3
9,2024-03-22,"2,024.00",2024-03,11.50,0.00,11.50,4,4


## Create business-friendly bins

Categorization is often useful for dashboards and communication.

For example, we may bucket revenue into bands:
- negative / return
- low
- medium
- high


In [29]:
def revenue_band(x):
    if pd.isna(x):
        return "Missing"
    elif x < 0:
        return "Return/Negative"
    elif x < 20:
        return "Low"
    elif x < 60:
        return "Medium"
    else:
        return "High"

df["revenue_band"] = df["revenue"].apply(revenue_band)
df["revenue_band"].value_counts()

revenue_band
Medium             2114
High               1586
Low                 699
Return/Negative      93
Missing              12
Name: count, dtype: int64

## Validate the cleaned dataset

A common failure in student work is to clean data but never verify the result.

A strong workflow always includes checks like:
- remaining missing values
- remaining duplicates
- final data types
- summary statistics after cleaning
- category levels after standardization


In [30]:
print("Remaining duplicates:", df.duplicated().sum())
print("\nRemaining missing values:")
print(df.isna().sum().sort_values(ascending=False))
print("\nData types:")
print(df.dtypes)

Remaining duplicates: 0

Remaining missing values:
segment             225
shipping_cost       197
city                123
channel              92
order_date           55
order_month          55
order_year           55
product_name         36
revenue              12
order_id              0
country               0
customer_id           0
category              0
region                0
discount              0
unit_price            0
quantity              0
is_zero_quantity      0
is_return             0
net_unit_price        0
abs_quantity          0
revenue_band          0
dtype: int64

Data types:
order_id                       str
order_date          datetime64[us]
customer_id                    str
segment                     string
channel                     string
country                     string
city                        string
region                      string
category                    string
product_name                string
quantity                     int64
unit_price

In [31]:
for col in ["country", "segment", "channel", "category", "region"]:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).head(15))


--- country ---
country
United States     1704
United Kingdom     851
Germany            559
Canada             550
Australia          432
France             408
Name: count, dtype: Int64

--- segment ---
segment
Consumer          2711
Small Business    1045
Corporate          523
<NA>               225
Name: count, dtype: Int64

--- channel ---
channel
Online          2706
Marketplace      903
Store Pickup     803
<NA>              92
Name: count, dtype: Int64

--- category ---
category
Coffee       1261
Bakery        924
Equipment     826
Merch         788
Tea           705
Name: count, dtype: Int64

--- region ---
region
NORTH AMERICA    2254
EUROPE           1818
APAC              432
Name: count, dtype: Int64


In [32]:
df[["quantity", "unit_price", "discount", "shipping_cost", "revenue", "net_unit_price"]].describe().T

,count,mean,std,min,25%,50%,75%,max
quantity,"4,504.00",2.91,1.54,-2.00,2.00,3.00,4.00,5.00
unit_price,"4,504.00",20.21,14.27,8.55,13.00,16.62,22.00,82.95
discount,"4,504.00",0.02,0.04,0.00,0.00,0.00,0.00,0.15
shipping_cost,"4,307.00",7.48,2.89,2.50,5.01,7.43,9.98,12.49
revenue,"4,492.00",58.29,55.89,-165.90,27.00,47.02,72.00,414.75
net_unit_price,"4,504.00",19.88,14.03,7.27,13.00,16.50,21.60,82.95


## Quick comparison: before vs after

It is useful to compare the raw and cleaned data to illustrate what changed.


In [33]:
comparison = pd.DataFrame({
    "raw_missing": raw_df.isna().sum(),
    "clean_missing": df.isna().sum().reindex(raw_df.columns, fill_value=np.nan)
})
comparison

,raw_missing,clean_missing
order_id,0,0
order_date,0,55
customer_id,0,0
segment,211,225
channel,79,92
country,0,0
city,103,123
region,0,0
category,0,0
product_name,36,36


### Question

What changed the most after cleaning?
- missingness?
- standardization?
- type correctness?
- row count?


## Lab

Students should now perform a mini-cleaning workflow on their own.

### Task set A — audit
1. Count missing values by column
2. Find duplicate rows
3. Inspect distinct values in `category`, `channel`, and `country`

### Task set B — clean
4. Convert placeholder strings to missing
5. Strip whitespace from text columns
6. Standardize `category` and `channel`
7. Convert `order_date` to datetime
8. Convert `unit_price` and `discount` to numeric

### Task set C — transform
9. Create `order_month`
10. Create `net_unit_price`
11. Create a return flag
12. Create a missing-shipping flag

### Task set D — validate
13. Show final missing-value counts
14. Show final data types
15. Summarize revenue by category


In [34]:
# Add your code below



## Solution sketch

Use after students attempt the work.


In [35]:
sol = raw_df.copy()

for col in ["city", "segment", "channel"]:
    sol[col] = sol[col].replace(["", " ", "unknown", "N/A", "n/a", "Unknown", "NULL", "null"], np.nan)

for col in ["country", "city", "segment", "channel", "region", "category", "product_name"]:
    sol[col] = sol[col].astype("string").str.strip()

sol["country"] = sol["country"].str.title()
sol["city"] = sol["city"].str.title()
sol["segment"] = sol["segment"].str.title().fillna("Unknown")
sol["channel"] = sol["channel"].str.title().fillna("Unknown")
sol["category"] = sol["category"].str.title()
sol["region"] = sol["region"].str.upper().replace({"Apac": "APAC"})

sol["unit_price"] = pd.to_numeric(sol["unit_price"].astype("string").str.replace("$", "", regex=False), errors="coerce")

disc = sol["discount"].astype("string").str.strip()
pct = disc.str.contains("%", na=False)
disc_num = pd.to_numeric(disc.str.replace("%", "", regex=False), errors="coerce")
disc_num = np.where(pct, disc_num / 100, disc_num)
sol["discount"] = pd.to_numeric(disc_num, errors="coerce")

sol["order_date"] = pd.to_datetime(sol["order_date"], errors="coerce")
sol = sol.drop_duplicates()

sol.loc[sol["shipping_cost"] < 0, "shipping_cost"] = np.nan
sol["is_return"] = sol["quantity"].lt(0)
sol["is_missing_shipping"] = sol["shipping_cost"].isna()
sol["shipping_cost"] = sol["shipping_cost"].fillna(sol["shipping_cost"].median())

recomputed = (sol["quantity"] * sol["unit_price"] * (1 - sol["discount"])).round(2)
sol["revenue"] = sol["revenue"].fillna(recomputed)

sol["order_month"] = sol["order_date"].dt.to_period("M").astype(str)
sol["net_unit_price"] = (sol["unit_price"] * (1 - sol["discount"])).round(2)

print(sol.isna().sum().sort_values(ascending=False).head(10))
print(sol.dtypes.head(10))
sol.groupby("category")["revenue"].sum().sort_values(ascending=False)

city            123
order_month      55
order_date       55
product_name     36
order_id          0
customer_id       0
channel           0
country           0
region            0
category          0
dtype: int64
order_id                   str
order_date      datetime64[us]
customer_id                str
segment                 string
channel                 string
country                 string
city                    string
region                  string
category                string
product_name            string
dtype: object


category
Equipment   99,569.77
Coffee      59,811.31
Merch       44,403.58
Bakery      32,350.40
Tea         26,268.29
Name: revenue, dtype: float64

## Case discussion: cleaning decisions are strategic

**"What is the correct way to clean the data?"**

The honest answer is:

> The best cleaning decision depends on the business problem, the downstream use case, and the meaning of the data.

Examples:
- For operational reporting, imputing some fields may be reasonable.
- For audit or finance work, you may need stricter rules.
- For machine learning, flags for missingness may be valuable.
- For customer-facing reports, category standardization is critical.

This is why data cleaning is both a **technical** and a **managerial** skill.


## Build a final analysis-ready table

A good session should end with a clear final output.

Let's choose a polished set of columns for later modules.


In [60]:
analysis_df = df[[
    "order_id", "order_date", "order_year", "order_month",
    "customer_id", "segment", "channel",
    "country", "city", "region",
    "category", "product_name",
    "quantity", "abs_quantity", "is_return", "is_zero_quantity",
    "unit_price", "discount", "net_unit_price", "shipping_cost",
    "revenue", "revenue_band"
]].copy()

analysis_df.head()

,order_id,order_date,order_year,order_month,customer_id,segment,channel,country,city,region,category,product_name,quantity,abs_quantity,is_return,is_zero_quantity,unit_price,discount,net_unit_price,shipping_cost,revenue,revenue_band
0,O100002,2024-04-01,"2,024.00",2024-04,C10061,Consumer,Online,Germany,Munich,EUROPE,Merch,Ceramic Mug,2,2,False,False,14.00,0.00,14.00,10.36,28.00,Medium
1,O100004,2024-05-07,"2,024.00",2024-05,C10147,Small Business,Marketplace,Australia,Brisbane,APAC,Merch,Tote Bag,3,3,False,False,18.00,0.00,18.00,10.78,54.00,Medium
2,O100005,2024-03-15,"2,024.00",2024-03,C10303,Corporate,Marketplace,Australia,Melbourne,APAC,Bakery,Biscotti Jar,2,2,False,False,11.50,0.00,11.50,NaN,23.00,Medium
3,O100006,2024-01-18,"2,024.00",2024-01,C10309,Small Business,Store Pickup,United States,San Jose,NORTH AMERICA,Coffee,Colombian Beans,4,4,False,False,16.50,0.00,16.50,6.87,66.00,High
4,O100009,2024-03-26,"2,024.00",2024-03,C10443,Small Business,Marketplace,France,Lyon,EUROPE,Tea,Matcha Tin,2,2,False,False,19.00,0.00,19.00,2.57,38.00,Medium


In [61]:
analysis_df.info()

<class 'pandas.DataFrame'>
Index: 4504 entries, 0 to 4509
Data columns (total 22 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   order_id          4504 non-null   str           
 1   order_date        4449 non-null   datetime64[us]
 2   order_year        4449 non-null   float64       
 3   order_month       4449 non-null   str           
 4   customer_id       4504 non-null   str           
 5   segment           4279 non-null   string        
 6   channel           4412 non-null   string        
 7   country           4504 non-null   string        
 8   city              4381 non-null   string        
 9   region            4504 non-null   string        
 10  category          4504 non-null   string        
 11  product_name      4468 non-null   string        
 12  quantity          4504 non-null   int64         
 13  abs_quantity      4504 non-null   int64         
 14  is_return         4504 non-null   bool  

## Example post-cleaning business summaries

Once the data is cleaned, it becomes much easier to produce reliable summaries.


In [62]:
analysis_df.groupby("category")["revenue"].agg(["count", "sum", "mean"]).sort_values("sum", ascending=False)

,count,sum,mean
category,,,
Equipment,824,"99,445.77",120.69
Coffee,1257,"59,600.11",47.41
Merch,786,"44,316.18",56.38
Bakery,921,"32,264.75",35.03
Tea,704,"26,228.29",37.26


In [63]:
analysis_df.groupby("order_month")["revenue"].sum()

order_month
2024-01   45,765.59
2024-02   40,784.73
2024-03   44,527.18
2024-04   41,258.76
2024-05   43,638.96
2024-06   42,946.81
Name: revenue, dtype: float64

In [64]:
analysis_df.groupby("channel")["revenue"].agg(["count", "sum", "mean"]).sort_values("sum", ascending=False)

,count,sum,mean
channel,,,
Online,2701,"159,914.47",59.21
Marketplace,898,"50,382.20",56.10
Store Pickup,802,"45,945.56",57.29


## Reflection

1. Which cleaning problems were easiest to spot?
2. Which problems required business judgment rather than pure coding?
3. When is it better to fill missing values versus drop rows?
4. What risks come from over-cleaning the data?
5. How would poor cleaning affect later visualization or machine learning modules?


## Challenging Exercises

- Create a column that flags rows where actual `revenue` differs materially from recomputed revenue.
- Standardize product names more aggressively.
- Create a reusable cleaning function that takes a raw DataFrame and returns a cleaned one.
- Build a data quality report table with:
    - column name
    - type
    - missing count
    - missing %
    - unique count
    - example values
- Design a cleaning log documenting every transformation step.


In [ ]:
# Extension 1
check_df = analysis_df.copy()
check_df["revenue_recomputed"] = (df["quantity"] * df["unit_price"] * (1 - df["discount"])).round(2)
check_df["revenue_gap"] = (check_df["revenue"] - check_df["revenue_recomputed"]).round(2)
check_df.loc[check_df["revenue_gap"].abs() > 0.01, ["order_id", "revenue", "revenue_recomputed", "revenue_gap"]].head(20)

In [ ]:
# Extension 3
def clean_retail_data(input_df):
    out = input_df.copy()

    for col in ["city", "segment", "channel"]:
        out[col] = out[col].replace(["", " ", "unknown", "N/A", "n/a", "Unknown", "NULL", "null"], np.nan)

    for col in ["country", "city", "segment", "channel", "region", "category", "product_name"]:
        out[col] = out[col].astype("string").str.strip()

    out["country"] = out["country"].str.title()
    out["city"] = out["city"].str.title()
    out["segment"] = out["segment"].str.title().fillna("Unknown")
    out["channel"] = out["channel"].str.title().fillna("Unknown")
    out["category"] = out["category"].str.title()
    out["region"] = out["region"].str.upper().replace({"Apac": "APAC"})

    out["unit_price"] = pd.to_numeric(out["unit_price"].astype("string").str.replace("$", "", regex=False), errors="coerce")

    disc = out["discount"].astype("string").str.strip()
    pct = disc.str.contains("%", na=False)
    disc_num = pd.to_numeric(disc.str.replace("%", "", regex=False), errors="coerce")
    disc_num = np.where(pct, disc_num / 100, disc_num)
    out["discount"] = pd.to_numeric(disc_num, errors="coerce")

    out["order_date"] = pd.to_datetime(out["order_date"], errors="coerce")
    out = out.drop_duplicates()

    out.loc[out["shipping_cost"] < 0, "shipping_cost"] = np.nan
    out["shipping_cost"] = out["shipping_cost"].fillna(out["shipping_cost"].median())

    recomputed = (out["quantity"] * out["unit_price"] * (1 - out["discount"])).round(2)
    out["revenue"] = out["revenue"].fillna(recomputed)

    out["is_return"] = out["quantity"].lt(0)
    out["is_zero_quantity"] = out["quantity"].eq(0)
    out["order_year"] = out["order_date"].dt.year
    out["order_month"] = out["order_date"].dt.to_period("M").astype(str)
    out["net_unit_price"] = (out["unit_price"] * (1 - out["discount"])).round(2)
    out["abs_quantity"] = out["quantity"].abs()

    return out

cleaned_again = clean_retail_data(raw_df)
cleaned_again.head()